<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

# Building a Demo

In this learning experience so far you have built the individual pieces of the lane-following pipeline, and in [the previous (FSM) notebook](05_finite_state_machine.ipynb) you saw how the **Finite State Machine** orchestrates which nodes
are active. In this final notebook we tie everything together into a runnable **demo**.

A "demo" in Duckietown is the top-level layer you launch on a robot with the `dts devel` workflow, which we have explored in the [the first (`dts devel`) notebook](00_DTS_devel_API.ipynb). We call it a "demo" (demonstration), because it's a single-command that gets a robot behavior started.

Building a demo means assembling four layers, all of which live in the **[`dt-ros2-core`](https://github.com/duckietown/dt-ros2-core)** repository (which is included here as a git submodule under `packages/dt-ros2-core`):

| Layer | Where it lives | What it does |
|-------|----------------|--------------|
| **Launcher** | `dt-ros2-core/launchers/<name>.sh` | The entrypoint `dts devel run -L` executes; calls `ros2 launch`. |
| **Demo launch file** | `dt-ros2-core/packages/duckietown_demo/launch/<name>.launch.py` | A Python launch file (package `duckietown_demos`) that starts the nodes this demo needs, in the robot's namespace. |
| **The nodes** | `dt-ros2-core/packages/<pkg>/` | The ROS 2 packages of the stack (`line_detector`, `ground_projection`, `lane_filter`, `lane_control`, `fsm`, `led_emitter`, ...), each with its `config/` files. |
| **FSM config** | `dt-ros2-core/packages/fsm/config/fsm_node/<name>.yaml` | The state machine for this demo, passed to the `fsm_node` by the launch file. |

We will build the launcher and the demo launch file from scratch, following the shape of the shipped `lane_following.launch.py`, and then generate the files into the `dt-ros2-core` submodule so the new demo can actually be launched with `dts devel run`.


## Launch files in ROS 2

In ROS 1, Duckietown put every subsystem into one big XML `master.launch` file behind on/off switches, and a demo was a thin file that included it. In ROS 2 a launch file is a **Python script** that defines a `generate_launch_description()` function returning a `LaunchDescription`: a list of *actions* such as `DeclareLaunchArgument` (the `veh:=...` arguments you pass on the command line), `Node` (start one executable of one package, with a name, parameters and topic remappings) and `GroupAction([PushRosNamespace(veh), ...])` (put everything inside it in the robot's namespace).

Have a look at the shipped [`lane_following.launch.py`](../packages/dt-ros2-core/packages/duckietown_demo/launch/lane_following.launch.py): it declares the `veh` and `param_file` arguments, then one `Node(...)` per component of the pipeline, e.g.

```python
line_detector_node = Node(
    package="line_detector",
    executable="line_detector_node.py",
    name="line_detector_node",
    output="screen",
    parameters=[line_detector_params],   # <share>/line_detector/config/line_detector_node/<param_file>
)
```

and groups them under `PushRosNamespace(veh)`. Two details are worth noticing:

* the wiring between nodes (which output topic feeds which input) is done with `remappings`, e.g. the ground projection's `lineseglist_in` is remapped to the line detector's `segment_list`;
* the FSM configuration is loaded by a small helper, `_load_yaml_parameters`, that reads the YAML file and serialises the nested maps (`states`, `events`, ...) into YAML strings, because ROS 2 parameters cannot hold dictionaries. The name of that file is what selects the state machine of the demo.

So, in ROS 2, "building a demo" means: **pick the nodes** you need from the stack (with their parameter files and remappings) and **pick the FSM configuration**. The rest of this notebook automates that.


## Finding the pieces

First, let's find the three directories we will be working with. The cell walks up from the notebook to locate the `dt-ros2-core` submodule, so the rest of the notebook works regardless of where it is run from.


In [ ]:
import os

def find_dt_ros2_core(start="."):
    """Walk up from `start` looking for the dt-ros2-core submodule checkout."""
    here = os.path.abspath(start)
    candidates = [
        # running from notebooks/ (the usual case)
        "../packages/dt-ros2-core",
        # running from the repo root
        "packages/dt-ros2-core",
        "lx/lx-lane-following/packages/dt-ros2-core",
    ]
    for rel in candidates:
        p = os.path.abspath(os.path.join(here, rel))
        if os.path.isdir(os.path.join(p, "packages", "duckietown_demo")):
            return p
    raise FileNotFoundError("Could not locate the dt-ros2-core submodule from " + here)

DT_ROS2_CORE = find_dt_ros2_core()
DEMOS_LAUNCH_DIR = os.path.join(DT_ROS2_CORE, "packages", "duckietown_demo", "launch")
LAUNCHERS_DIR    = os.path.join(DT_ROS2_CORE, "launchers")
FSM_CONFIG_DIR   = os.path.join(DT_ROS2_CORE, "packages", "fsm", "config", "fsm_node")
REFERENCE_LAUNCH = os.path.join(DEMOS_LAUNCH_DIR, "lane_following.launch.py")

print("dt-ros2-core:   ", DT_ROS2_CORE)
print("demo launches:  ", os.path.relpath(DEMOS_LAUNCH_DIR, DT_ROS2_CORE))
print("launchers:      ", os.path.relpath(LAUNCHERS_DIR, DT_ROS2_CORE))
print("fsm configs:    ", os.path.relpath(FSM_CONFIG_DIR, DT_ROS2_CORE))
assert os.path.exists(REFERENCE_LAUNCH), "lane_following.launch.py not found!"
print("\nFound lane_following.launch.py \u2713")


## Step 1 — see what nodes the reference demo starts

A ROS 2 launch file is Python, so we can parse it with the standard `ast` module and list every `Node(...)` it creates: its package, executable and node name. These are the building blocks available to our demo.


In [ ]:
import ast

def list_launch_nodes(launch_path):
    """Return [(package, executable, name)] for every Node(...) call in a Python launch file."""
    tree = ast.parse(open(launch_path).read(), filename=launch_path)
    found = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Call) and getattr(node.func, "id", None) == "Node":
            kw = {k.arg: k.value for k in node.keywords}
            def const(key):
                v = kw.get(key)
                return v.value if isinstance(v, ast.Constant) else None
            found.append((const("package"), const("executable"), const("name")))
    return found

reference_nodes = list_launch_nodes(REFERENCE_LAUNCH)
print(f"{len(reference_nodes)} nodes started by lane_following.launch.py:\n")
print(f"  {'node name':26s} {'package':20s} executable")
for package, executable, name in reference_nodes:
    print(f"  {name:26s} {package:20s} {executable}")


## Step 2 — write the demo launch file

A demo launch file is a list of `Node(...)` actions in the robot's namespace. The cell below defines a small **catalog** of the nodes of the lane following stack, each as the Python snippet that starts it (package, executable, name, parameter file and remappings copied from `lane_following.launch.py`), and a helper that assembles a complete launch file from a selection of them. Compare its output with the existing
[`lane_following.launch.py`](../packages/dt-ros2-core/packages/duckietown_demo/launch/lane_following.launch.py),
they have the same shape.

Note the `fsm_file_name` argument: it selects the FSM configuration (`packages/fsm/config/fsm_node/<fsm_file_name>.yaml`) that the `fsm_node` entry loads. Our example reuses the existing `lane_following` state machine so that it works without you having to also build a new FSM config (see notebook 05 for that). The optional `node_params` lets a demo override parameters of a node right in the launch file, which is how we make our example demo *slow*.


In [ ]:
# One entry per node of the lane following stack. `__EXTRA__` is where per-demo parameter overrides go.
NODE_CATALOG = {
    "line_detector_node": '''
    nodes.append(Node(
        package="line_detector", executable="line_detector_node.py", name="line_detector_node", output="screen",
        parameters=[PathJoinSubstitution([get_package_share_directory("line_detector"),
                                          "config", "line_detector_node", param_file])__EXTRA__],
    ))''',
    "ground_projection_node": '''
    nodes.append(Node(
        package="ground_projection", executable="ground_projection_node", name="ground_projection_node", output="screen",
        parameters=[os.path.join(get_package_share_directory("ground_projection"),
                                 "config", "ground_projection_node", "default.yaml")__EXTRA__],
        remappings=[("lineseglist_in", "segment_list")],          # input: the line detector's segments
    ))''',
    "lane_filter_node": '''
    nodes.append(Node(
        package="lane_filter", executable="lane_filter_node", name="lane_filter_node", output="screen",
        parameters=[PathJoinSubstitution([get_package_share_directory("lane_filter"),
                                          "config", "lane_filter_node", param_file])__EXTRA__],
        remappings=[("segment_list", "lineseglist_out")],         # input: the ground-projected segments
    ))''',
    "lane_controller_node": '''
    nodes.append(Node(
        package="lane_control", executable="lane_controller_node", name="lane_controller_node", output="screen",
        parameters=[__EXTRA__],
    ))''',
    "fsm_node": '''
    nodes.append(Node(
        package="fsm", executable="fsm_node", name="fsm_node", output="screen",
        parameters=_load_yaml_parameters("fsm", "config/fsm_node", "__FSM_FILE__.yaml"),
        remappings=[
            ("fsm_node/set_pattern", "set_pattern"),              # LED pattern requests go to the LED emitter
            ("mode_override", "joy_mapper_node/mode_override"),
            ("car_cmd_in", "car_cmd"),
        ],
    ))''',
    "led_emitter_node": '''
    nodes.append(Node(
        package="led_emitter", executable="led_emitter_node", name="led_emitter_node", output="screen",
        parameters=[{
            "protocol_file": os.path.join(get_package_share_directory("led_emitter"),
                                          "config", "led_emitter_node", "LED_protocol.yaml"),
            "LED_scale": 0.8,
            "robot_type": "duckiebot",
        }__EXTRA__],
    ))''',
}

LAUNCH_HEADER = '''#!/usr/bin/env python3
"""{demo_name} demo: generated by the lx-lane-following notebook 06."""
import os

import yaml
from ament_index_python.packages import get_package_share_directory
from launch import LaunchDescription
from launch.actions import DeclareLaunchArgument, GroupAction
from launch.substitutions import LaunchConfiguration, PathJoinSubstitution
from launch_ros.actions import Node, PushRosNamespace


def _load_yaml_parameters(package_name, config_path, param_file):
    """Load a ROS 2 YAML parameter file for Node(parameters=...).

    Nested maps/lists are serialised to YAML strings: ROS 2 parameters cannot hold dictionaries.
    """
    full_path = os.path.join(get_package_share_directory(package_name), config_path, param_file)
    with open(full_path, "r") as fh:
        raw = yaml.safe_load(fh)
    if "/**" in raw:
        ros_params = raw["/**"]["ros__parameters"]
    elif "ros__parameters" in raw:
        ros_params = raw["ros__parameters"]
    else:
        ros_params = raw
    processed = {{}}
    for key, value in ros_params.items():
        processed[key] = yaml.dump(value, default_flow_style=False) if isinstance(value, (dict, list)) else value
    return [processed]


def generate_launch_description():
    veh = LaunchConfiguration("veh")
    param_file = LaunchConfiguration("param_file")
    nodes = []
'''

LAUNCH_FOOTER = '''
    return LaunchDescription([
        DeclareLaunchArgument("veh", default_value=os.getenv("VEHICLE_NAME", ""),
                              description="Robot name, used as the ROS namespace"),
        DeclareLaunchArgument("param_file", default_value="{param_file}",
                              description="Parameter file name (without path) for line_detector and lane_filter"),
        GroupAction([PushRosNamespace(veh), *nodes]),
    ])
'''


def build_demo_launch(demo_name, node_names, fsm_file_name, param_file="default.yaml", node_params=None):
    """Assemble a demo launch file from the catalog entries in `node_names`."""
    node_params = node_params or {}
    unknown = [n for n in node_names if n not in NODE_CATALOG]
    if unknown:
        raise ValueError(f"unknown nodes (not in NODE_CATALOG): {unknown}")
    out = [LAUNCH_HEADER.format(demo_name=demo_name)]
    for name in node_names:
        snippet = NODE_CATALOG[name].replace("__FSM_FILE__", fsm_file_name)
        overrides = node_params.get(name)
        if "__EXTRA__" in snippet:
            if overrides:
                # append an in-line parameter dict after the parameter file (if any)
                base_present = "[__EXTRA__]" not in snippet
                extra = (", " if base_present else "") + repr(dict(overrides))
            else:
                extra = ""
            snippet = snippet.replace("__EXTRA__", extra)
        elif overrides:
            raise ValueError(f"{name} does not take parameter overrides")
        out.append(snippet)
    out.append(LAUNCH_FOOTER.format(param_file=param_file))
    return "".join(out).rstrip() + "\n"


# Our example: a slower lane-following variant. Same nodes as lane_following, reusing the existing
# 'lane_following' FSM config, with the nominal speed of the controller lowered in the launch file.
DEMO_NAME = "lane_following_slow"
DEMO_NODES = [
    "line_detector_node",
    "ground_projection_node",
    "lane_filter_node",
    "lane_controller_node",
    "fsm_node",
    "led_emitter_node",
]

demo_launch_py = build_demo_launch(
    DEMO_NAME, DEMO_NODES, fsm_file_name="lane_following",
    node_params={"lane_controller_node": {"v_bar": 0.15}},
)
print(demo_launch_py)


Before writing anything, let's make sure every node we picked actually exists in the reference demo, i.e. that the catalog is consistent with what `lane_following.launch.py` starts (a typo here would make `ros2 launch` fail on the robot), and that the generated file is valid Python.


In [ ]:
reference_names = {name for _, _, name in reference_nodes}
unknown = [n for n in DEMO_NODES if n not in reference_names]
if unknown:
    print("These nodes are NOT started by lane_following.launch.py:")
    for n in unknown:
        print("  ", n)
else:
    print("All", len(DEMO_NODES), "nodes are known from lane_following.launch.py \u2713")

# the generated launch file must at least compile, and start the nodes we asked for
compile(demo_launch_py, f"{DEMO_NAME}.launch.py", "exec")
print("Generated launch file compiles \u2713")


## Step 3 — write the launcher

The launcher is the script `dts devel run` executes. It is a thin bash wrapper that runs `ros2 launch` on the demo
launch file of the `duckietown_demos` package, passing the robot name as the `veh` argument. The pattern (boilerplate + a single `dt-exec ros2 launch` line) matches the existing
[`lane-following.sh`](../packages/dt-ros2-core/launchers/lane-following.sh).

**Naming convention:** launcher files use hyphens (`lane-following.sh`) while launch files use underscores
(`lane_following.launch.py`). The launcher's base name (without `.sh`) is what you pass to `dts devel run -L`.


In [ ]:
def build_launcher(launch_file_basename):
    return f"""#!/bin/bash

source /environment.sh

# initialize launch file
dt-launchfile-init

# YOUR CODE BELOW THIS LINE
# ----------------------------------------------------------------------------

# launching app (ROS 2)
dt-exec ros2 launch duckietown_demos {launch_file_basename}.launch.py veh:="${{VEHICLE_NAME}}"

# ----------------------------------------------------------------------------
# YOUR CODE ABOVE THIS LINE

# wait for app to end
dt-launchfile-join
"""

# launcher name uses hyphens; it launches the underscore-named .launch.py file
LAUNCHER_NAME = DEMO_NAME.replace("_", "-")  # 'lane-following-slow'
launcher_sh = build_launcher(DEMO_NAME)
print(launcher_sh)


## Step 4 — generate the files into `dt-ros2-core`

Now we write the two files to their real locations in the submodule. The cell refuses to overwrite an
existing file (so you don't clobber a shipped demo), set `OVERWRITE = True` if you really mean to. The
launcher is made executable, just like the others. The `duckietown_demos` package installs every file of its `launch/` directory, so the new launch file is picked up by the next `dts devel build` without touching any build file.


In [ ]:
import stat

OVERWRITE = False

def write_file(path, contents, executable=False):
    if os.path.exists(path) and not OVERWRITE:
        print(f"SKIP (exists): {path}")
        return
    with open(path, "w") as f:
        f.write(contents)
    if executable:
        st = os.stat(path)
        os.chmod(path, st.st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    print(f"WROTE: {path}")

launch_path   = os.path.join(DEMOS_LAUNCH_DIR, f"{DEMO_NAME}.launch.py")
launcher_path = os.path.join(LAUNCHERS_DIR, f"{LAUNCHER_NAME}.sh")

write_file(launch_path, demo_launch_py)
write_file(launcher_path, launcher_sh, executable=True)


## Step 5 — validate the whole chain

Finally, let's confirm the pieces are consistent: the launcher must reference a launch file that exists, the
launch file must be valid Python defining `generate_launch_description`, and we warn if the FSM config the demo points at is missing (the
node would fail to start on the robot otherwise).


In [ ]:
import re

def validate_demo(launcher_path):
    problems, notes = [], []
    if not os.path.exists(launcher_path):
        problems.append(f"launcher not found: {launcher_path}")
        return problems, notes
    launcher = open(launcher_path).read()
    m = re.search(r"ros2 launch\s+duckietown_demos\s+(\S+)\.launch\.py", launcher)
    if not m:
        problems.append("launcher does not `ros2 launch` a duckietown_demos launch file")
        return problems, notes
    launch_base = m.group(1)
    notes.append(f"launcher -> {launch_base}.launch.py")
    lp = os.path.join(DEMOS_LAUNCH_DIR, launch_base + ".launch.py")
    if not os.path.exists(lp):
        problems.append(f"launch file referenced by launcher is missing: {lp}")
        return problems, notes
    launch_src = open(lp).read()
    try:
        compile(launch_src, lp, "exec")
    except SyntaxError as e:
        problems.append(f"{launch_base}.launch.py is not valid Python: {e}")
        return problems, notes
    if "def generate_launch_description" not in launch_src:
        problems.append(f"{launch_base}.launch.py does not define generate_launch_description()")
    started = [name for _, _, name in list_launch_nodes(lp)]
    notes.append(f"nodes started -> {', '.join(started) if started else 'none!'}")
    if not started:
        problems.append(f"{launch_base}.launch.py starts no Node")
    # which FSM config will this demo use?
    fm = re.search(r'_load_yaml_parameters\(\s*"fsm",\s*"config/fsm_node",\s*"([^"]+)\.yaml"', launch_src)
    if fm is None:
        notes.append("FSM config -> none (no fsm_node in this demo)")
    else:
        fsm = fm.group(1)
        fsm_path = os.path.join(FSM_CONFIG_DIR, fsm + ".yaml")
        if os.path.exists(fsm_path):
            notes.append(f"FSM config -> {fsm}.yaml \u2713")
        else:
            notes.append(f"FSM config -> {fsm}.yaml  (MISSING - build one per notebook 05)")
    return problems, notes

problems, notes = validate_demo(launcher_path)
for n in notes:
    print("  ", n)
print()
if problems:
    print("Problems found:")
    for p in problems:
        print("  -", p)
else:
    print("Demo chain looks consistent \u2713")


## Step 6 — build and run your demo

Creating the files is only half of it, now you build and run the demo using the same `dts devel` workflow
introduced in [the first notebook](00_DTS_devel_API.ipynb). All of these commands run from the `dt-ros2-core` project
directory:

```bash
cd packages/dt-ros2-core
```

**1. Build** so your new launcher and launch file are baked into the image (`ROBOT_NAME` is your real or
virtual robot, with the ROS 2 stack up):

```bash
dts devel build -H ROBOT_NAME
```

**2. Run** by passing the launcher's base name (no `.sh`) to `-L`:

```bash
dts devel run -H ROBOT_NAME -L lane-following-slow
```

That runs `launchers/lane-following-slow.sh`, which runs `ros2 launch duckietown_demos lane_following_slow.launch.py`, which starts the six nodes of the lane following stack in your robot's namespace, bringing up the FSM (using the `lane_following` config) and the lane following pipeline. You can check with `ros2 node list` from a terminal in the code editor, and with `ros2 param get /ROBOT_NAME/lane_controller_node v_bar` that the override from the launch file (0.15 m/s) is in effect.

Because the example reuses the lane following stack and FSM config, `lane_following_slow` behaves just like
lane following, only slower; the point is the *scaffolding*: change the nodes, the parameter files and overrides, or the FSM config and
you have a brand new demo.


## Recap & exercise

Building a demo is assembling four layers, all in the `dt-ros2-core` submodule:

1. The **nodes** of the stack already exist as ROS 2 packages (`packages/<pkg>`), with their `config/` files; you rarely touch them.
2. A **demo launch file** (`packages/duckietown_demo/launch/<name>.launch.py`) starts the nodes you need in the robot's namespace, wires them with remappings and passes them their parameters.
3. A **launcher** (`launchers/<name>.sh`) runs `ros2 launch duckietown_demos <name>.launch.py`; its base name is what you pass to `dts devel run -L`.
4. An **FSM config** (notebook 05) defines the states; the launch file selects it by file name.

### Exercise

Build your own demo end-to-end:

1. Pick a `DEMO_NAME` and a set of nodes (try a demo that only runs the perception part, `line_detector_node`, `ground_projection_node` and `lane_filter_node`, and look at `lane_filter_node/lane_pose` while you push the robot by hand; or add parameter overrides for the line detector, e.g. `{"line_detector_node": {"top_cutoff": 50}}`).
2. Create a matching FSM config `packages/fsm/config/fsm_node/<DEMO_NAME>.yaml` using what you learned in notebook 05
   (give it a `LANE_FOLLOWING` and a `STOP` state driven by the obstacle events), and pass `fsm_file_name=DEMO_NAME` so the demo uses *your* config.
3. Re-run Steps 2-5. The validator should report your launcher, launch file, and FSM config all line up.

When `validate_demo` prints no problems and resolves to *your* FSM config, you've built a demo from scratch.
